# Lab 13 - Charlie Caravati

In [5]:
import time
import matplotlib.pyplot as plt
import numpy as np
from numpy.linalg import inv
from numpy.linalg import norm
from scipy.linalg import lu_factor, lu_solve

In [35]:
def driver(delta):

    # fix the random seed so every run gives the same results
    np.random.seed(1)

# example 1
    '''
    n = 1000

    dd = np.ones(n)
    dd[0] = 4000
    dd[1] = 4000-500
    dd[2] = 10   
    '''
# example 2
    '''
    n = 4
# example 3: |3+4i| = |3-4i| = 5, no dominant eigenvalue -> power method fails
#    dd = np.array([-2, 1, 3+4j, 3-4j])
    '''

# example 3
    n = 5
    dd = np.array([10, 20, 20.001, 80,50])
#    dd = np.arange(1, n+1)
    
#    print('dd = ', dd)
    
    X = np.random.randn(n, n)

    # X*diag(dd)*inv(X): eigenvalues are the entries of dd
    A = X @ np.diag(dd) @ inv(X)

    # shift: inverse iteration converges to the eigenvalue closest to alpha
    alpha = 19+delta

    # target eigenvalue and convergence rate |lambda_1-alpha|/|lambda_2-alpha|
    # where lambda_1, lambda_2 are the closest and second closest to alpha
    dist = np.sort(np.abs(dd - alpha))
    lam1 = dd[np.argmin(np.abs(dd - alpha))]
    rate = dist[0]/dist[1]
    print('target eigenvalue = ', lam1)
    print('rate = ', rate)

    tol = 1e-8
    t = time.perf_counter()
    [lambdas, X] = np.linalg.eig(A)
    print('eig time: %g seconds' % (time.perf_counter() - t))
    print('lambdas = ', lambdas)

    Nmax = 1500 
    t = time.perf_counter()
    [lam, v,it] = shiftInvpowerIt(A,alpha, Nmax,tol)
    print(f'For alpha {alpha}')
    print('shifted inverse power time: %g seconds' % (time.perf_counter() - t))
    print('shifted inverse power number of iterations:', it)
    print('shifted inverse power: eigenvalue found:', lam[it-1])

    # only keep the iterations actually used
    err = np.abs(lam[:it] - lam1)

    N = np.arange(1, it+1)

    # plt.semilogy(N, err, 'o-', label=r'$|\lambda^{(k)} - \lambda_1|$')
    # plt.semilogy(N, rate**N, 'r', label=r'$\left(|\lambda_1-\alpha|/|\lambda_2-\alpha|\right)^k$')
    # plt.xlabel('iteration $k$')
    # plt.ylabel('error')
    # plt.title(r'Shifted inverse power iteration, $\alpha = %g$' % alpha)
    # plt.legend()
    # plt.show()

In [3]:
def powerIt(A, Nmax,tol):

    n = A.shape[0]

    q = np.random.rand(n)
    q = q/np.linalg.norm(q)
    lam = np.zeros(Nmax, dtype=complex)

    for j in range(Nmax):
        z = A @ q
        q = z/np.linalg.norm(z)
        lam[j] = np.vdot(q, A @ q)
#        if (np.abs(lam[j]-lam[j-1])/np.abs(lam[j])<tol):
        if (np.abs(lam[j]-lam[j-1])<tol):
           return [lam, q,j+1]

    v = q

    return [lam, v,j+1]

In [50]:
def shiftInvpowerIt(A,alpha, Nmax,tol):

    n = A.shape[0]

    # factor A - alpha*I once; each iteration is then just two triangular solves
    LU = lu_factor(A-alpha*np.identity(n))
    
#    print('LU = ', LU)

    q = np.random.rand(n)
    q = q/np.linalg.norm(q)
    q0 = q
    lam = np.zeros(Nmax, dtype=complex)

    for j in range(Nmax):
        z = lu_solve(LU, q0)
        q = z/np.linalg.norm(z)
        q1 = q
        # Rayleigh quotient with A (not the inverse) gives the eigenvalue of A
        lam[j] = np.vdot(q, A @ q)
        if (j > 0 and np.abs(lam[j]-lam[j-1])/np.abs(lam[j])<tol):
        # if (j > 0 and norm(q1-q0)<tol):
           return [lam, q,j+1]
        q0=q1   
#        if (j > 0 and np.abs(lam[j]-lam[j-1])<tol):
#           return [lam, q,j+1]

    v = q

    return [lam, v,j+1]

In [37]:
dd = np.array([10, 20, 20.001, 80,50])

In [51]:
driver(.5)

target eigenvalue =  20.0
rate =  0.9980039920159657
eig time: 6.84583e-05 seconds
lambdas =  [80.    50.    10.    20.001 20.   ]
For alpha 19.5
shifted inverse power time: 0.0103849 seconds
shifted inverse power number of iterations: 609
shifted inverse power: eigenvalue found: (20.000026059236372+0j)


In [52]:
driver(.9)

target eigenvalue =  20.0
rate =  0.9900990099009782
eig time: 6.13332e-05 seconds
lambdas =  [80.    50.    10.    20.001 20.   ]
For alpha 19.9
shifted inverse power time: 0.00381638 seconds
shifted inverse power number of iterations: 181
shifted inverse power: eigenvalue found: (19.99999606686816+0j)


In [53]:
driver(.99)

target eigenvalue =  20.0
rate =  0.909090909090821
eig time: 8.14581e-05 seconds
lambdas =  [80.    50.    10.    20.001 20.   ]
For alpha 19.99
shifted inverse power time: 0.000771 seconds
shifted inverse power number of iterations: 24
shifted inverse power: eigenvalue found: (19.99999208719044+0j)


In [54]:
driver(.999)

target eigenvalue =  20.0
rate =  0.5
eig time: 8.9e-05 seconds
lambdas =  [80.    50.    10.    20.001 20.   ]
For alpha 19.999
shifted inverse power time: 0.000586125 seconds
shifted inverse power number of iterations: 10
shifted inverse power: eigenvalue found: (19.999999843894184+0j)


In [55]:
driver(1-1e-10)

target eigenvalue =  20.0
rate =  1.0000177463039835e-07
eig time: 0.000143708 seconds
lambdas =  [80.    50.    10.    20.001 20.   ]
For alpha 19.999999999899998
shifted inverse power time: 0.000246125 seconds
shifted inverse power number of iterations: 2
shifted inverse power: eigenvalue found: (19.99999999999985+0j)


The number of iterations decreases as delta increases, which brings alpha closer to the actual eigenvalue. In the pre-lab, alpha was decreasing but was similarly moving in a direction towards the true value so that number of iterations decreased as alpha approach 4, similar to alpha approaching 20.

In [56]:
def shiftInvpowerIt(A,alpha, Nmax,tol):

    n = A.shape[0]

    # factor A - alpha*I once; each iteration is then just two triangular solves
    LU = lu_factor(A-alpha*np.identity(n))
    
#    print('LU = ', LU)

    q = np.random.rand(n)
    q = q/np.linalg.norm(q)
    q0 = q
    lam = np.zeros(Nmax, dtype=complex)

    for j in range(Nmax):
        z = lu_solve(LU, q0)
        q = z/np.linalg.norm(z)
        q1 = q
        # Rayleigh quotient with A (not the inverse) gives the eigenvalue of A
        lam[j] = np.vdot(q, A @ q)
        # if (j > 0 and np.abs(lam[j]-lam[j-1])/np.abs(lam[j])<tol):
        if (j > 0 and norm(q1-q0)<tol):
           return [lam, q,j+1]
        q0=q1   
#        if (j > 0 and np.abs(lam[j]-lam[j-1])<tol):
#           return [lam, q,j+1]

    v = q

    return [lam, v,j+1]

In [57]:
driver(.5)

target eigenvalue =  20.0
rate =  0.9980039920159657
eig time: 0.000188125 seconds
lambdas =  [80.    50.    10.    20.001 20.   ]
For alpha 19.5
shifted inverse power time: 0.0257517 seconds
shifted inverse power number of iterations: 1500
shifted inverse power: eigenvalue found: (19.999993971931314+0j)


In [58]:
driver(.9)

target eigenvalue =  20.0
rate =  0.9900990099009782
eig time: 9.99589e-05 seconds
lambdas =  [80.    50.    10.    20.001 20.   ]
For alpha 19.9
shifted inverse power time: 0.01863 seconds
shifted inverse power number of iterations: 1378
shifted inverse power: eigenvalue found: (19.999999999821796+0j)


In [59]:
driver(.99)

target eigenvalue =  20.0
rate =  0.909090909090821
eig time: 4.71668e-05 seconds
lambdas =  [80.    50.    10.    20.001 20.   ]
For alpha 19.99
shifted inverse power time: 0.00252 seconds
shifted inverse power number of iterations: 168
shifted inverse power: eigenvalue found: (19.99999999998212+0j)


In [63]:
driver(.999)

target eigenvalue =  20.0
rate =  0.5
eig time: 6.37923e-05 seconds
lambdas =  [80.    50.    10.    20.001 20.   ]
For alpha 19.999
shifted inverse power time: 0.00103262 seconds
shifted inverse power number of iterations: 27
shifted inverse power: eigenvalue found: (19.999999999998753+0j)


In [62]:
driver(1-1e-10)

target eigenvalue =  20.0
rate =  1.0000177463039835e-07
eig time: 0.000105459 seconds
lambdas =  [80.    50.    10.    20.001 20.   ]
For alpha 19.999999999899998
shifted inverse power time: 0.000129625 seconds
shifted inverse power number of iterations: 3
shifted inverse power: eigenvalue found: (19.999999999999883+0j)


Just using the a stopping criteria of the norm between q1 and q0 below tolerance appears much stricter. There are more iterations for each delta that using relative error.

In [73]:
def shiftInvpowerIt(A,alpha, Nmax,tol):

    n = A.shape[0]

    # factor A - alpha*I once; each iteration is then just two triangular solves
    LU = lu_factor(A-alpha*np.identity(n))
    
#    print('LU = ', LU)

    q = np.random.rand(n)
    q = q/np.linalg.norm(q)
    q0 = q
    lam = np.zeros(Nmax, dtype=complex)

    for j in range(Nmax):
        z = lu_solve(LU, q0)
        q = z/np.linalg.norm(z)
        q1 = q
        # Rayleigh quotient with A (not the inverse) gives the eigenvalue of A
        lam[j] = np.vdot(q, A @ q)
        # if (j > 0 and np.abs(lam[j]-lam[j-1])/np.abs(lam[j])<tol):
        # # if (j > 0 and norm(q1-q0)<tol):
        #    return [lam, q,j+1]
        q0=q1   
        if (j > 0 and np.abs(lam[j]-lam[j-1])<tol):
          return [lam, q,j+1]

    v = q

    return [lam, v,j+1]

In [75]:
driver(.99)

target eigenvalue =  20.0
rate =  0.909090909090821
eig time: 0.000123667 seconds
lambdas =  [80.    50.    10.    20.001 20.   ]
For alpha 19.99
shifted inverse power time: 0.000962833 seconds
shifted inverse power number of iterations: 25
shifted inverse power: eigenvalue found: (19.99999209160799+0j)


In [74]:
driver(.999)

target eigenvalue =  20.0
rate =  0.5
eig time: 8.34172e-05 seconds
lambdas =  [80.    50.    10.    20.001 20.   ]
For alpha 19.999
shifted inverse power time: 0.000388375 seconds
shifted inverse power number of iterations: 14
shifted inverse power: eigenvalue found: (19.999999990199683+0j)


We tried a few different stopping conditions including using norms and absolute error as suggested but I could not decrease iterations myself.